# Week 3, Bonus: A Neural Network Forecaster

**A tiny brain takes on the market.**

*Author: The Genius Project Year 3*

---

### What you will build
- A small **neural network** that forecasts from a window of recent returns.
- A fair fight against a plain linear model.
- A grounded view of what deep learning does &mdash; and does not &mdash; buy you.

### The big idea, in one breath
A neural network stacks many simple "neurons" so it can bend and twist to fit patterns
a straight line cannot. It is the same idea behind the AI that writes and draws. Here
we build a small one with scikit-learn and point it at the stock.

> Real AI labs use **TensorFlow** or **PyTorch** for the biggest networks (you met
> TensorFlow in the Week 2 bonus). The `MLPClassifier` below is the same core idea in a
> few lines, so you can focus on how it *thinks* rather than the plumbing.

### Step 1: Build a window of features

We describe each day by the **last 10 daily returns** &mdash; a sliding window &mdash;
and ask the network to predict whether tomorrow closes up or down. More history per
example gives the network more to chew on.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

stock = pd.read_csv("data/daily_stock.csv", parse_dates=["date"]).set_index("date")
stock["return"] = stock["price"].pct_change()

WINDOW = 10
# Each feature is the return some number of days ago.
for k in range(1, WINDOW + 1):
    stock[f"ret_{k}"] = stock["return"].shift(k)

stock["up_tomorrow"] = (stock["return"].shift(-1) > 0).astype(int)

feat_cols = [f"ret_{k}" for k in range(1, WINDOW + 1)]
d = stock.dropna(subset=feat_cols + ["up_tomorrow"])
X = d[feat_cols]
y = d["up_tomorrow"]
print("Examples:", len(X), "| features per example:", len(feat_cols))

### Step 2: Scale the features

Neural networks learn best when every input is on a similar scale. A
`StandardScaler` re-centres each feature to mean 0 and spread 1. We **fit the scaler
on the training data only**, so no information about the future leaks into training.

In [ ]:
from sklearn.preprocessing import StandardScaler

TEST = int(len(X) * 0.2)
X_train, X_test = X.iloc[:-TEST], X.iloc[-TEST:]
y_train, y_test = y.iloc[:-TEST], y.iloc[-TEST:]

scaler = StandardScaler().fit(X_train)     # learn the scale from training days only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

### Step 3: Train the neural network

`MLPClassifier` is a **multi-layer perceptron** &mdash; a classic neural network. Our
`hidden_layer_sizes=(16, 8)` means two hidden layers, one with 16 neurons and one with
8. We line it up against a plain linear model and the always-guess baseline.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

baseline = int(y_train.mean() >= 0.5)
baseline_acc = accuracy_score(y_test, [baseline] * len(y_test))

linear = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
net = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=1000,
                    early_stopping=True, random_state=42).fit(X_train_s, y_train)

print(f"Baseline (always guess {'up' if baseline else 'down'}): {baseline_acc * 100:.1f}%")
print(f"Linear model:                {accuracy_score(y_test, linear.predict(X_test_s)) * 100:.1f}%")
print(f"Neural network:              {accuracy_score(y_test, net.predict(X_test_s)) * 100:.1f}%")

### Step 4: Watch the network learn

As it trains, the network repeatedly nudges its neurons to make fewer mistakes. Its
**loss** (a measure of wrongness) should fall and then flatten &mdash; the shape of a
model learning all it can from the data.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(net.loss_curve_, color="#8172B3")
plt.title("The neural network's error as it trains")
plt.xlabel("Training round")
plt.ylabel("Loss (lower = fewer mistakes)")
plt.grid(alpha=0.3)
plt.show()

### The honest takeaway

The neural network is far more powerful than a straight line &mdash; yet on this stock
it barely edges the baseline, just like the simpler models in Notebook 5. When the
signal is nearly absent (remember: returns have almost no memory), a bigger brain finds
almost nothing more. Power cannot invent a pattern that is not there.

That is the most valuable lesson of the whole week. The magic of machine learning is
real, but it lives in the **data and the features**, not in the size of the model. Give
these same tools a predictable problem &mdash; your monthly spending, a seasonal sales
chart &mdash; and they shine.

### What you learned
- A **neural network** stacks simple neurons to fit patterns a line cannot.
- **Feature scaling** helps a network learn.
- More model power does **not** rescue a near-random problem.
- The edge in ML comes from good data and good features.

### Where to go next
- Rebuild this network in **TensorFlow/Keras** (from the Week 2 bonus) &mdash; same idea,
  the tool real labs use.
- Point everything you learned at **real** data: a public budgeting export, or a real
  price history. The workflow &mdash; describe, simulate, check stationarity, forecast,
  model, and stay honest about baselines &mdash; is exactly the one professionals use.

### The whole week, in one line
Describe your data, simulate the uncertain future, respect stationarity, forecast
against a baseline, and let machine learning find the patterns that are truly there.